In [2]:
# Import all basic functions
import onnx
import onnxruntime as ort
import ml_dtypes
import numpy as np
import h5py
import torch
import tqdm
from execute.io_onnx import run_iobinding_cpu_input_gpu_output
import os
from construct.construct import ensure_base_model, ensure_extra_outputs
import onnxruntime as ort

# Import config
from config import load_config

# data generation
from generate_data.datagen import (
    sample_ambient_points,
    sample_around_points,
    repeat_points_to_match,
)

# model generation
from construct.construct import (
    ensure_base_model,
    ensure_subgraph_pullback,
    ensure_subgraphs,
)


# execution code
from execute.optimization_loop import optimization_loop, NewtonRaphson
import importlib
import sys

In [3]:
# Load config
cfg = load_config("/nfs-share/pa511/code_bases/new_jac/configs/simple_stories.toml")

# Re-use all existing models
force = False


# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)

Base model already exists, skipping export -> onnx_models/simple_stories/base_model.onnx


In [ ]:
with h5py.File("data/simple_stories.h5", "r") as h5f:
    # print([i for i in h5f["input_ids"]])
    base_seqs = h5f["input_ids"][:]


[array([  29, 2390, 2760,  134,   65,  214, 1725,  146]), array([   0,  357, 2390, 2760,  134,   65,   20,   27]), array([  85, 1003,  407, 1580, 1012,  224, 2548,  108]), array([ 120,  876,  130, 1843,   60,   99, 2065,  142]), array([  29,   29, 1003, 3609,  137,   29,   29,    1]), array([ 140,  130, 1618,   77, 1431, 2390, 2760,  118]), array([  85, 1003,    8,   50, 2557, 1474,  126,  171]), array([ 262, 3982,   60,  490,  698,   77,  463,  305]), array([  85, 1003, 2810,  604, 2326,   85, 2815,   36]), array([ 140,   85, 1647, 1850,  734,  146, 3833,  280]), array([ 770,  145,   35,  449,   93, 1483, 2750,   60]), array([  29,   29, 1580, 1012,  224, 2548,   29,   29]), array([ 486,  235,  699,  915,  158, 2390, 2760,  134]), array([  85,  304,   83,   95,  442,  115, 1217,   87]), array([  29,   29,   29, 1106,   29,   29,   29,    1]), array([  85, 1106,  127, 1369, 3784,  410, 1916,  189]), array([  29,   29,   29, 4008,  480,   29,   29,   29]), array([108, 278, 699, 482,  87

In [5]:
n_samples = 1000


In [6]:
def build_chunk_rows(
    cfg,
    base_seqs: np.ndarray,
    row_start: int,
    row_end: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    # Total samples
    n_samples = base_seqs.shape[0]
    
    # Gets global index for big dataset
    row_sample_idx, row_pos, row_vocab_id = np.unravel_index(
        np.arange(row_start, row_end), (n_samples, 8, 4019),
    )
    # Copy base sequences to return value
    token_ids = base_seqs[row_sample_idx].copy()  # (chunk, cfg.seq)
    
    token_ids[np.arange(row_end - row_start), row_pos] = row_vocab_id
    print(row_end, row_pos[0])
    return token_ids, row_sample_idx, row_pos, row_vocab_id



In [8]:
base_seqs = base_seqs[:n_samples]
n_rows = n_samples * 8 * 4019

n_chunks = -(-n_rows // 32)  # ceil division

ensure_base_model(cfg, force=False, verbose=False)
extra_outputs = [sublayer.output for sublayer in cfg.sublayers]
model_path = ensure_extra_outputs(cfg, extra_outputs, force=False)


sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
sess_options.intra_op_num_threads = 8

out_sess = ort.InferenceSession(
    str(model_path),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)

out_output_names = [o.name for o in out_sess.get_outputs()]
out_input_names = [o.name for o in out_sess.get_inputs()]
print(out_output_names, out_input_names)
out_path = "test/"
os.makedirs(os.path.dirname(out_path), exist_ok=True)



Base model already exists, skipping export -> onnx_models/simple_stories/base_model.onnx
Extra outputs model already exists, skipping generation -> onnx_models/simple_stories/base_model_extra_outputs.onnx
['output', 'embedding', 'add_6', 'add_8', 'add_12', 'add_14', 'add_18', 'add_20', 'add_24', 'add_26', 'add_30', 'add_32', 'add_36', 'add_38'] ['input_ids']


In [ ]:
for c in tqdm.tqdm(range(n_chunks)):
    row_start = c * 32
    row_end = min(row_start + 32, n_rows)
    valid = row_end - row_start
    # row_start, row_end -- which rows in the big perturbed dataset are we generating now
    token_ids, chunk_sample_idx, chunk_pos, chunk_vocab_id = build_chunk_rows(
        None,
        base_seqs,
        row_start,
        row_end,
    )
    # break
    print(token_ids)
    
    fx = run_iobinding_cpu_input_gpu_output(
                    out_sess,
                    out_input_names[0],
                    token_ids,
                    out_output_names,
                )
    print(fx)
    break



  0%|          | 0/1004750 [00:00<?, ?it/s]

32 0
[[   0 2390 2760  134   65  214 1725  146]
 [   1 2390 2760  134   65  214 1725  146]
 [   2 2390 2760  134   65  214 1725  146]
 [   3 2390 2760  134   65  214 1725  146]
 [   4 2390 2760  134   65  214 1725  146]
 [   5 2390 2760  134   65  214 1725  146]
 [   6 2390 2760  134   65  214 1725  146]
 [   7 2390 2760  134   65  214 1725  146]
 [   8 2390 2760  134   65  214 1725  146]
 [   9 2390 2760  134   65  214 1725  146]
 [  10 2390 2760  134   65  214 1725  146]
 [  11 2390 2760  134   65  214 1725  146]
 [  12 2390 2760  134   65  214 1725  146]
 [  13 2390 2760  134   65  214 1725  146]
 [  14 2390 2760  134   65  214 1725  146]
 [  15 2390 2760  134   65  214 1725  146]
 [  16 2390 2760  134   65  214 1725  146]
 [  17 2390 2760  134   65  214 1725  146]
 [  18 2390 2760  134   65  214 1725  146]
 [  19 2390 2760  134   65  214 1725  146]
 [  20 2390 2760  134   65  214 1725  146]
 [  21 2390 2760  134   65  214 1725  146]
 [  22 2390 2760  134   65  214 1725  146]
 [  23

  0%|          | 0/1004750 [00:00<?, ?it/s]

[tensor([[[  9.5178,   3.1914,   1.1959,  ...,   0.8094,   1.8773,   1.1982],
         [  1.0811,   3.2817,   4.0756,  ...,  -0.4563,   2.7047,   1.3495],
         [ -7.0829,   0.3169,   1.8109,  ...,   0.5361,  -1.1934,  -1.2797],
         ...,
         [ -0.5230,   2.5198,   4.3173,  ...,   1.3538,   0.7659,   2.4667],
         [ -4.5879,   6.1103,   7.6114,  ...,  -0.7564,  -0.1170,  -0.0927],
         [ -7.4220,   2.8426,   4.2008,  ...,  -2.8033,  -0.2954,  -2.7470]],

        [[ -6.6550,  -4.0567,  -3.4020,  ...,  -4.5079,  -3.9845,  -3.4146],
         [  3.2232,   4.6566,   2.1853,  ...,   2.0369,   4.6644,   3.6056],
         [ -9.7767,  -1.1366,  -0.1992,  ...,  -3.4533,  -0.9557,  -1.3517],
         ...,
         [ -0.1359,   0.8990,   2.6180,  ...,   1.1920,   1.5084,   2.8019],
         [ -4.5513,   3.2543,   3.7771,  ...,   1.3695,   0.2621,   1.9319],
         [ -8.1554,   1.9364,  -0.2945,  ...,  -1.7025,   0.2310,  -2.2109]],

        [[-12.2749,   4.3009,  -1.3699,  ..

In [11]:
fx[0]

tensor([[[  9.5178,   3.1914,   1.1959,  ...,   0.8094,   1.8773,   1.1982],
         [  1.0811,   3.2817,   4.0756,  ...,  -0.4563,   2.7047,   1.3495],
         [ -7.0829,   0.3169,   1.8109,  ...,   0.5361,  -1.1934,  -1.2797],
         ...,
         [ -0.5230,   2.5198,   4.3173,  ...,   1.3538,   0.7659,   2.4667],
         [ -4.5879,   6.1103,   7.6114,  ...,  -0.7564,  -0.1170,  -0.0927],
         [ -7.4220,   2.8426,   4.2008,  ...,  -2.8033,  -0.2954,  -2.7470]],

        [[ -6.6550,  -4.0567,  -3.4020,  ...,  -4.5079,  -3.9845,  -3.4146],
         [  3.2232,   4.6566,   2.1853,  ...,   2.0369,   4.6644,   3.6056],
         [ -9.7767,  -1.1366,  -0.1992,  ...,  -3.4533,  -0.9557,  -1.3517],
         ...,
         [ -0.1359,   0.8990,   2.6180,  ...,   1.1920,   1.5084,   2.8019],
         [ -4.5513,   3.2543,   3.7771,  ...,   1.3695,   0.2621,   1.9319],
         [ -8.1554,   1.9364,  -0.2945,  ...,  -1.7025,   0.2310,  -2.2109]],

        [[-12.2749,   4.3009,  -1.3699,  ...

In [12]:
out_output_names

['output',
 'embedding',
 'add_6',
 'add_8',
 'add_12',
 'add_14',
 'add_18',
 'add_20',
 'add_24',
 'add_26',
 'add_30',
 'add_32',
 'add_36',
 'add_38']

In [15]:
valid = 32
batch_arange = np.arange(valid)


In [16]:
for i, output_name in enumerate(out_output_names[1:], start=1):
    print(i, output_name)
    layer_out = fx[i][:valid]  # (valid, cfg.seq, d_model)
    swept_vecs = (
        layer_out[batch_arange, chunk_pos]
        .to(torch.float32)
        .detach()
        .cpu()
        .numpy()
    )

1 embedding
2 add_6
3 add_8
4 add_12
5 add_14
6 add_18
7 add_20
8 add_24
9 add_26
10 add_30
11 add_32
12 add_36
13 add_38
